# Semantic Search System Analysis

This notebook demonstrates the semantic search system with Pinecone, GMM clustering, and semantic caching.

In [ ]:
import sys
sys.path.insert(0, '/vercel/share/v0-project/src')

from embeddings import EmbeddingManager
from clustering import GMMClusterer
from semantic_cache import SemanticCache
from data_pipeline import DataPipeline
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

print("Imports successful!")

## 1. Load and Prepare Data

In [ ]:
# Initialize pipeline
pipeline = DataPipeline()

# Load 20 Newsgroups
texts, categories = pipeline.load_20newsgroups(subset="train", remove_headers=True)

# Sample for faster processing
texts = pipeline.sample_texts(texts, n_samples=2000)

# Preprocess
texts, valid_indices = pipeline.preprocess_texts(texts)

print(f"\nLoaded {len(texts)} documents")
print(f"Sample text: {texts[0][:200]}...")

## 2. Generate Embeddings

In [ ]:
# Initialize embedding manager
embedding_manager = EmbeddingManager()

# Generate embeddings
embeddings = embedding_manager.encode(texts)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embedding_manager.embedding_dim}")

## 3. Fit GMM Clustering

In [ ]:
# Initialize and fit GMM
gmm_clusterer = GMMClusterer(n_components=20)
cluster_metrics = gmm_clusterer.fit(embeddings)

print(f"\nClustering metrics:")
for key, value in cluster_metrics.items():
    print(f"  {key}: {value}")

# Get cluster assignments
cluster_labels = gmm_clusterer.predict(embeddings)
print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for cluster_id, count in zip(unique, counts):
    print(f"  Cluster {cluster_id}: {count} documents")

## 4. Get Cluster Keywords

In [ ]:
# Get representative keywords for each cluster
cluster_keywords = gmm_clusterer.get_cluster_keywords(embeddings, texts, n_keywords=10)

print("Cluster Keywords:")
for cluster_id in range(gmm_clusterer.n_components):
    keywords = cluster_keywords.get(cluster_id, [])
    if keywords:
        print(f"  Cluster {cluster_id:2d}: {', '.join(keywords)}")

## 5. Visualize Clusters with PCA

In [ ]:
# Apply PCA for visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Create scatter plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], 
                       c=cluster_labels, cmap='tab20', alpha=0.6, s=30)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.title('Document Clusters Visualization (PCA)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"\nPCA explained variance: {sum(pca.explained_variance_ratio_):.1%}")

## 6. Test Semantic Cache

In [ ]:
# Initialize semantic cache
cache = SemanticCache(similarity_threshold=0.85)

# Simulate queries
test_queries = [
    "what is machine learning",
    "machine learning algorithms",  # Similar to above - should hit cache
    "climate change effects",
    "global warming impacts",  # Similar to above
    "sports news today"
]

print("Simulating cache queries:")
for i, query in enumerate(test_queries):
    # Generate embedding
    query_embedding = embedding_manager.encode_single(query)
    
    # Get cluster
    cluster_id, confidence = gmm_clusterer.get_dominant_cluster(query_embedding)
    
    # Lookup in cache
    result, similarity = cache.lookup(query_embedding, cluster_id)
    
    if result is None:
        # Add to cache (simulated result)
        simulated_result = {"query_id": i, "status": "found"}
        cache.add(query, query_embedding, simulated_result, cluster_id)
        hit_status = "MISS (added to cache)"
    else:
        hit_status = f"HIT (similarity: {similarity:.3f})"
    
    print(f"  {i+1}. '{query}' -> Cluster {cluster_id} [{hit_status}]")

# Print cache stats
stats = cache.get_stats()
print(f"\nCache Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## 7. Test Query Embeddings Similarity

In [ ]:
# Compare similarity between similar queries
query1 = "machine learning"
query2 = "machine learning algorithms"
query3 = "sports news"

emb1 = embedding_manager.encode_single(query1)
emb2 = embedding_manager.encode_single(query2)
emb3 = embedding_manager.encode_single(query3)

sim_12 = embedding_manager.cosine_similarity(emb1, emb2)
sim_13 = embedding_manager.cosine_similarity(emb1, emb3)
sim_23 = embedding_manager.cosine_similarity(emb2, emb3)

print("Embedding Similarity Comparisons:")
print(f"  '{query1}' vs '{query2}': {sim_12:.4f}")
print(f"  '{query1}' vs '{query3}': {sim_13:.4f}")
print(f"  '{query2}' vs '{query3}': {sim_23:.4f}")

## Summary

This notebook demonstrates:
- Loading 20 Newsgroups dataset
- Generating embeddings with Sentence Transformers
- Fitting Gaussian Mixture Model for soft clustering
- Analyzing cluster characteristics
- Visualizing clusters with PCA
- Testing semantic cache functionality
- Comparing query embedding similarities